# 02 · Exploratory Data Analysis – STATS19 2023

Reads the interim merged file produced by `01-load-merge.py` and generates  
one histogram per feature (coloured by severity class) plus a class-distribution chart.

**Run after:** `01-load-merge.py`  
**Outputs:** `outputs/figures/fig-hist-<feature>-2023-v1.png`, `fig-severity-distribution-2023-v1.png`

In [ ]:
# --- Paths ---
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT   = Path("..").resolve().parent  # src/notebooks/ -> repo root
INTERIM     = REPO_ROOT / "data" / "interim"
OUTPUTS_FIG = REPO_ROOT / "outputs" / "figures"
OUTPUTS_FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# --- Load the merged interim file ---
df = pd.read_csv(INTERIM / "stats19-collision-vehicle-casualty-2023-interim-v1.csv", low_memory=False)
print(f"Loaded {len(df):,} rows")
df.head(3)

In [ ]:
# --- Map numeric severity codes to readable labels ---
# STATS19 codes: 1 = Fatal, 2 = Serious, 3 = Slight
SEVERITY_MAP     = {1: "Fatal", 2: "Serious", 3: "Slight"}
SEVERITY_PALETTE = {"Slight": "#4C9BE8", "Serious": "#F4A261", "Fatal": "#E76F51"}

df["severity_label"] = df["casualty_severity"].map(SEVERITY_MAP)

print("Class distribution:")
print(df["severity_label"].value_counts())

In [ ]:
# --- Bar chart showing how many casualties per severity class ---
counts = df["severity_label"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values,
       color=[SEVERITY_PALETTE[s] for s in counts.index])
ax.set_title("Casualty severity distribution - STATS19 2023")
ax.set_ylabel("Count")
for i, (lbl, val) in enumerate(zip(counts.index, counts.values)):
    ax.text(i, val + 200, f"{val:,}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUTS_FIG / "fig-severity-distribution-2023-v1.png", dpi=150)
plt.show()

In [ ]:
# --- One histogram per feature, coloured by severity class ---
FEATURE_COLS = [
    "road_type", "speed_limit", "weather_conditions", "light_conditions",
    "road_surface_conditions", "day_of_week", "number_of_vehicles", "vehicle_type",
]
existing = [c for c in FEATURE_COLS if c in df.columns]

for feat in existing:
    fig, ax = plt.subplots(figsize=(9, 4))

    if df[feat].nunique() <= 20:
        # Few unique values -> grouped bar chart
        plot_df = (
            df.groupby([feat, "severity_label"])
            .size()
            .reset_index(name="count")
        )
        sns.barplot(data=plot_df, x=feat, y="count", hue="severity_label",
                    palette=SEVERITY_PALETTE, ax=ax)
        ax.tick_params(axis="x", rotation=45)
    else:
        # Many unique values -> overlapping histogram
        for sev, grp in df.groupby("severity_label"):
            ax.hist(grp[feat].dropna(), bins=30, alpha=0.6,
                    label=sev, color=SEVERITY_PALETTE.get(sev, "grey"))
        ax.legend()

    ax.set_title(f"{feat} by severity")
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(OUTPUTS_FIG / f"fig-hist-{feat}-2023-v1.png", dpi=150)
    plt.close(fig)
    print(f"Saved: fig-hist-{feat}-2023-v1.png")

In [ ]:
# --- Quick overview of missing values for all feature columns ---
null_summary = df[existing + ["casualty_severity"]].isnull().sum().rename("null_count").to_frame()
null_summary["pct"] = (null_summary["null_count"] / len(df) * 100).round(2)
print(null_summary)